# Bronze — Yahoo Finance Ingestion
Fetches daily market data for ISK exchange rates and the OMX Iceland Index.

**Source:** Yahoo Finance via `yfinance`  
**Tickers:** `EURUSD=X`, `ISKUSD=X`, `^OMX`  
**Output:** `bronze.yahoo_finance_raw` (Delta table)

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import timedelta

TICKERS = ["EURUSD=X", "ISKUSD=X", "^OMX"]
START_DATE_HISTORY = "1990-01-01"
REBUILD_HISTORY = True 

if spark.catalog.tableExists("bronze.yahoo_finance_raw") and not REBUILD_HISTORY:
    last_date = spark.sql("SELECT MAX(date) FROM bronze.yahoo_finance_raw").collect()[0][0]
    start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
    is_full_load = False
    print(f"Incremental load — fetching from {start_date}")
else:
    start_date = START_DATE_HISTORY
    is_full_load = True
    print(f"Full load — fetching history from {start_date}")

In [ ]:
def sanitize_yf_dataframe(df):
    if df.empty: return df
    new_cols = []
    for col in df.columns.values:
        combined = "_".join(str(c) for c in col) if isinstance(col, tuple) else str(col)
        clean_name = (combined.replace('=', '_')
                              .replace('^', '')
                              .replace('.', '_')
                              .replace(' ', '_')
                              .lower()
                              .strip()
                              .rstrip('_'))
        new_cols.append(clean_name)
    df.columns = new_cols
    df = df.reset_index()
    df.columns = [c.lower() for c in df.columns] 
    df["ingested_at"] = pd.Timestamp.now()
    return df

In [ ]:
raw_data = yf.download(TICKERS, start=start_date, interval="1d", group_by='column')

if not raw_data.empty:
    raw_data = sanitize_yf_dataframe(raw_data)
    print(f"Fetched {len(raw_data)} rows.")
else:
    mssparkutils.notebook.exit("no-data")

spark_df = spark.createDataFrame(raw_data)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.yahoo_finance_raw")
    print("Full history rebuild complete.")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.yahoo_finance_raw")
    print(f"Appended {spark_df.count()} records.")